# Phase 2: Data Quality Engineering — Rule-Based Duplicate & Validity Detection
### Grower Impact & Data Quality Dashboard — Greenstand Treetracker

**Author:** Jueeli Sawant
**Data source:** pulled via `data_access.py` -- currently `synthetic`, switches to `live` automatically once dev DB access is confirmed and `_load_captures_live()` is filled in. Nothing below needs to change.

**Objective (per proposal):** identify likely duplicate or invalid capture records using deterministic, explainable rules -- same grower + GPS proximity + timestamp window for duplicates, plus basic validity checks (implausible capture counts per grower per window, missing/null critical fields). Quantify the scale of the issue.


In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_access import load_captures, load_planters, DATA_SOURCE
import os
os.makedirs('../outputs', exist_ok=True)  # created automatically if missing

captures = load_captures()
planters = load_planters()

print("Data source:", DATA_SOURCE)
print("Captures:", captures.shape)
captures.head()

Data source: synthetic
Captures: (8095, 9)


,capture_id,planter_id,captured_at,lat,lon,species,country_name,verification_status,survived
0,cap_0,79,2023-02-27 17:29:00,8.445409,-13.308160,Rhizophora mucronata (Red Mangrove),Sierra Leone,verified,1
1,cap_1,409,2023-02-13 18:40:00,17.176577,-88.534847,Mango,Belize,verified,1
2,cap_2,262,2024-01-14 20:01:00,20.549832,78.853501,Moringa,India,verified,1
3,cap_3,307,2023-01-08 08:27:00,3.823636,11.474384,Acacia,Cameroon,verified,1
4,cap_4,485,2024-07-12 03:57:00,1.429407,32.316837,Cashew,Uganda,verified,1


## 1. Duplicate Detection

Rounding GPS to 5 decimal places (~1.1m precision) and truncating timestamp to the minute is a coarse grid match -- fine for a first pass, but it has a known blind spot: two captures 61 seconds apart, or GPS that differs in the 5th decimal only because of device jitter, won't match. The pandas version below implements the *intent* of the rule (same planter, GPS within a real-world distance tolerance, timestamps within a real time window) rather than the exact SQL grid, since that's closer to what actually catches duplicates in practice. Both approaches are shown so the SQL can be run directly against the live DB once access is confirmed, with this notebook as the validation/tuning ground.

In [2]:
# --- SQL-equivalent grid match (fast, coarse) ---
grid = captures.copy()
grid['lat_r'] = grid['lat'].round(5)
grid['lon_r'] = grid['lon'].round(5)
grid['minute'] = grid['captured_at'].dt.floor('min')

grid_dupes = (
    grid.groupby(['planter_id', 'lat_r', 'lon_r', 'minute'])
    .size()
    .reset_index(name='dup_count')
    .query('dup_count > 1')
)
print(f"SQL-grid method: {grid_dupes['dup_count'].sum()} records across "
      f"{len(grid_dupes)} duplicate groups "
      f"({grid_dupes['dup_count'].sum() / len(captures):.2%} of all captures)")

SQL-grid method: 36 records across 18 duplicate groups (0.44% of all captures)


In [3]:
# --- Proximity + time-window method (more realistic, catches near-misses the grid misses) ---
from math import radians
import numpy as np

def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in meters between two lat/lon points (vectorized)."""
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

GPS_TOLERANCE_M = 15       # captures within 15m of each other, same planter
TIME_WINDOW_MIN = 10       # and within 10 minutes of each other

df = captures.sort_values(['planter_id', 'captured_at']).reset_index(drop=True)
flagged_pairs = []

for planter_id, group in df.groupby('planter_id'):
    group = group.reset_index(drop=True)
    n = len(group)
    if n < 2:
        continue
    for i in range(n):
        for j in range(i + 1, n):
            t_diff = abs((group.loc[j, 'captured_at'] - group.loc[i, 'captured_at']).total_seconds() / 60)
            if t_diff > TIME_WINDOW_MIN:
                break  # sorted by time within planter, so no later j can be closer
            dist = haversine_m(group.loc[i, 'lat'], group.loc[i, 'lon'],
                                group.loc[j, 'lat'], group.loc[j, 'lon'])
            if dist <= GPS_TOLERANCE_M:
                flagged_pairs.append({
                    'planter_id': planter_id,
                    'capture_id_a': group.loc[i, 'capture_id'],
                    'capture_id_b': group.loc[j, 'capture_id'],
                    'distance_m': round(dist, 2),
                    'minutes_apart': round(t_diff, 2),
                })

dup_pairs_df = pd.DataFrame(flagged_pairs)
flagged_capture_ids = set(dup_pairs_df['capture_id_a']).union(set(dup_pairs_df['capture_id_b'])) if len(dup_pairs_df) else set()
print(f"Proximity+time method: {len(dup_pairs_df)} flagged pairs, "
      f"{len(flagged_capture_ids)} unique captures involved "
      f"({len(flagged_capture_ids) / len(captures):.2%} of all captures)")
dup_pairs_df.head(10)

Proximity+time method: 61 flagged pairs, 122 unique captures involved (1.51% of all captures)


,planter_id,capture_id_a,capture_id_b,distance_m,minutes_apart
0,1,cap_burst_5,cap_burst_9,10.7,4.0
1,15,cap_1610,cap_1610_dup,0.0,0.0
2,45,cap_2745,cap_2745_dup,0.0,1.0
3,58,cap_6977,cap_6977_dup,0.0,0.0
4,62,cap_7169,cap_7169_dup,0.0,1.0
5,64,cap_1907,cap_1907_dup,0.0,2.0
6,66,cap_3611,cap_3611_dup,0.0,1.0
7,70,cap_1576,cap_1576_dup,0.0,1.0
8,78,cap_1532,cap_1532_dup,0.0,2.0
9,82,cap_7535,cap_7535_dup,0.0,1.0


**Note on runtime:** the nested loop above is O(n²) *within each planter*, which is fine at this scale (8K rows, ~16 captures/planter on average) but would need a spatial index (e.g., a KD-tree via `scipy.spatial.cKDTree`, or PostGIS `ST_DWithin` directly in SQL) before running against a much larger live dataset. Flagging this now so it's not a surprise later -- worth switching to the indexed version once real data volume is known.

## 2. Validation Against Known Injected Duplicates

In [4]:
# The synthetic generator seeded 60 intentional near-duplicate rows, marked with '_dup' in
# capture_id purely for this validation step -- production data won't have this marker, which
# is exactly why the detection logic above needs to work without relying on it.
known_dup_ids = set(captures.loc[captures['capture_id'].str.contains('_dup'), 'capture_id'])
caught = known_dup_ids & flagged_capture_ids
missed = known_dup_ids - flagged_capture_ids

print(f"Known injected duplicates: {len(known_dup_ids)}")
print(f"Caught by proximity+time rule: {len(caught)} ({len(caught)/len(known_dup_ids):.1%})")
print(f"Missed: {len(missed)}")
if missed:
    print("\nMissed IDs (worth checking why -- likely a distance or time edge case):")
    print(captures[captures['capture_id'].isin(missed)][['capture_id','planter_id','captured_at','lat','lon']])

Known injected duplicates: 60
Caught by proximity+time rule: 60 (100.0%)
Missed: 0


## 3. Validity Checks

Per proposal: implausible capture counts per grower per time window, and missing/null critical fields.

In [5]:
# Implausible burst: too many captures from one planter in a short window.
# Threshold choice: median inter-capture gap globally is used as a sanity baseline,
# then anything with 10+ captures inside 60 minutes gets flagged as implausible --
# a real single grower physically registering that many trees that fast is unlikely
# and matches the injected 'burst' scenario (35 captures in ~35 min).
BURST_WINDOW_MIN = 60
BURST_MIN_COUNT = 10

burst_flags = []
for planter_id, group in df.groupby('planter_id'):
    group = group.sort_values('captured_at').reset_index(drop=True)
    ts = group['captured_at']
    for i in range(len(group)):
        window_end = ts[i] + pd.Timedelta(minutes=BURST_WINDOW_MIN)
        count_in_window = ((ts >= ts[i]) & (ts <= window_end)).sum()
        if count_in_window >= BURST_MIN_COUNT:
            burst_flags.append(planter_id)
            break

burst_flags = sorted(set(burst_flags))
print(f"Planters with an implausible capture burst (>= {BURST_MIN_COUNT} captures in "
      f"{BURST_WINDOW_MIN} min): {burst_flags}")

known_burst_planter = captures.loc[captures['capture_id'].str.contains('_burst'), 'planter_id'].unique()
print(f"Known injected burst planter: {list(known_burst_planter)}")
print(f"Rule caught it: {set(known_burst_planter).issubset(set(burst_flags))}")

Planters with an implausible capture burst (>= 10 captures in 60 min): [1]
Known injected burst planter: [np.int64(1)]
Rule caught it: True


In [6]:
# Missing/null critical fields -- captures side
critical_fields = ['planter_id', 'lat', 'lon', 'captured_at', 'species']
missing_report = captures[critical_fields].isnull().sum()
missing_report = missing_report[missing_report > 0]
print("Missing critical fields in captures:")
print(missing_report if len(missing_report) else "None -- synthetic captures are fully populated on these fields.")
print()
print("(Real data is unlikely to be this clean -- re-run this cell first against live data,")
print(" since a jump here is the single most likely early surprise.)")

Missing critical fields in captures:
None -- synthetic captures are fully populated on these fields.

(Real data is unlikely to be this clean -- re-run this cell first against live data,
 since a jump here is the single most likely early surprise.)


## 4. Quantified Impact Summary

In [7]:
total = len(captures)
dup_count = len(flagged_capture_ids)
burst_planter_count = len(burst_flags)

summary = pd.DataFrame({
    'Metric': [
        'Total captures analyzed',
        'Flagged as likely duplicate (proximity + time rule)',
        'Duplicate rate',
        'Planters flagged for implausible capture bursts',
        'Recall on known injected duplicates',
    ],
    'Value': [
        f"{total:,}",
        f"{dup_count:,}",
        f"{dup_count/total:.2%}",
        f"{burst_planter_count}",
        f"{len(caught)}/{len(known_dup_ids)} ({len(caught)/len(known_dup_ids):.0%})",
    ]
})
summary

,Metric,Value
0,Total captures analyzed,"8,095"
1,Flagged as likely duplicate (proximity + time ...,122
2,Duplicate rate,1.51%
3,Planters flagged for implausible capture bursts,1
4,Recall on known injected duplicates,60/60 (100%)


In [8]:
dup_pairs_df.to_csv('../outputs/phase2_flagged_duplicate_pairs.csv', index=False)
print("Saved flagged pairs to ../outputs/phase2_flagged_duplicate_pairs.csv")

Saved flagged pairs to ../outputs/phase2_flagged_duplicate_pairs.csv


## 5. Findings Summary

- **Duplicate rule performance:** the proximity+time rule (15m / 10min) catches essentially all of the known injected duplicates on synthetic data -- see recall above. This is a *synthetic-data validation*, not a claim about real-world recall; live data will have messier GPS noise and legitimate close-together captures (e.g., a grower planting a small grove in one visit) that the tolerance needs tuning against.
- **Tolerance is a live variable, not a fixed answer:** 15m/10min were chosen as reasonable starting points, not derived from real device GPS accuracy or real grower workflow patterns. Once live data is available, this should be re-tuned by manually reviewing a sample of flagged pairs (per the proposal's validation approach) rather than trusting the synthetic-data recall number.
- **Burst detection catches the injected anomaly cleanly** at the 10-captures/60-minutes threshold -- this threshold is also a placeholder pending real distribution of grower capture cadence.
- **SQL-grid vs. proximity+time:** the coarse SQL-grid match (mirroring the proposal's sample query) catches fewer duplicates than the proximity+time approach, since it requires an exact-minute + 5-decimal GPS match. Worth deciding whether to ship the simpler SQL version for transparency/speed, or the pandas version for recall -- possibly SQL for a first-pass flag and pandas for the deeper validation pass.
- **Missing critical fields:** none in this synthetic set -- flagged as the most likely place real data will differ meaningfully from what's tested here.
- **Next step:** once live data access is confirmed, re-run this notebook unchanged (`DATA_SOURCE` flips automatically), pull a manual review sample of ~30-50 flagged pairs, and use that to finalize the GPS/time tolerances before quantifying "X% of captures flagged as likely duplicates" for the actual findings report.
